# TUGAS AKHIR
## Analisis Prediksi Kanker Kolorektal dengan Machine Learning

**Dataset:** Colorectal Cancer Dataset

**Tahapan:**
1. **Data Processing** - Pembersihan, missing values, outlier, normalisasi
2. **Exploratory Data Analysis (EDA)** - Analisis eksploratif komprehensif
3. **Target Analysis** - Perbandingan semua target yang mungkin
4. **Training Importance Fitur** - Feature Importance + 4 Model (SVM, XGBoost, RF, KNN)
5. **Advanced Solutions** - NN, Polynomial, PCA, Stacking, GridSearch

In [ ]:
# ============================================================
# IMPORT LIBRARIES & SETUP
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import time
import gc

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier as RF, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    r2_score, mean_squared_error
)
from imblearn.over_sampling import SMOTE

import os as _os
_os.environ['OMP_NUM_THREADS'] = '2'
from xgboost import XGBClassifier
from scipy.stats import chi2_contingency

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

print('Semua library berhasil diimport!')
print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')
print('Scikit-learn: OK')

---
## BAGIAN 1: DATA PROCESSING

Tahapan preprocessing:
1. Cleaning dataset (duplikat, whitespace, validasi range)
2. Handling missing values
3. Remove outliers (IQR method)
4. Normalisasi & Standarisasi fitur

In [ ]:
# ================================================================
# 1.1 LOAD DATA AWAL
# ================================================================
print('=' * 70)
print('BAGIAN 1: DATA PROCESSING PIPELINE')
print('=' * 70)

df_raw = pd.read_csv('colorectal_cancer_dataset_asli.csv', sep=';')
print(f'\nDataset awal: {df_raw.shape[0]:,} baris, {df_raw.shape[1]} kolom')
print(f'Kolom: {list(df_raw.columns)}')

num_cols = ['Age', 'Tumor_Size_mm', 'Healthcare_Costs', 'Incidence_Rate_per_100K', 'Mortality_Rate_per_100K']
cat_cols = [c for c in df_raw.columns if c not in num_cols and c != 'Patient_ID']

df_raw.head()

In [ ]:
# ================================================================
# 1.2 CLEANING DATASET
# ================================================================
print('=' * 70)
print('TAHAP 1.2: CLEANING DATASET')
print('=' * 70)

df1 = df_raw.copy()
cleaning_log = []

n_duplicates = df1.duplicated().sum()
if n_duplicates > 0:
    df1 = df1.drop_duplicates()
    cleaning_log.append(f'Hapus {n_duplicates} baris duplikat')
print(f'  Duplikat: {n_duplicates} baris >> {"dihapus" if n_duplicates > 0 else "tidak ada"}')

for col in cat_cols:
    if col in df1.columns:
        has_ws = df1[col].astype(str).str.contains(r'^\s+|\s+$', na=False).sum()
        if has_ws > 0:
            df1[col] = df1[col].astype(str).str.strip()
            cleaning_log.append(f'Strip whitespace kolom {col} ({has_ws} nilai)')

invalid_age = ((df1['Age'] <= 0) | (df1['Age'] > 120)).sum()
if invalid_age > 0:
    df1 = df1[(df1['Age'] > 0) & (df1['Age'] <= 120)]
    cleaning_log.append(f'Hapus {invalid_age} baris dengan Age tidak valid')

invalid_tumor = (df1['Tumor_Size_mm'] <= 0).sum()
if invalid_tumor > 0:
    df1 = df1[df1['Tumor_Size_mm'] > 0]
    cleaning_log.append(f'Hapus {invalid_tumor} baris dengan Tumor_Size_mm <= 0')

invalid_cost = (df1['Healthcare_Costs'] <= 0).sum()
if invalid_cost > 0:
    df1 = df1[df1['Healthcare_Costs'] > 0]
    cleaning_log.append(f'Hapus {invalid_cost} baris dengan Healthcare_Costs <= 0')

print(f'\n  Hasil cleaning: {df1.shape[0]:,} baris, {df1.shape[1]} kolom')
for log in cleaning_log:
    print(f'    - {log}')

In [ ]:
# ================================================================
# 1.3 HANDLING MISSING VALUES
# ================================================================
print('=' * 70)
print('TAHAP 1.3: HANDLING MISSING VALUES')
print('=' * 70)

df2 = df1.copy()
mv_log = []

missing_before = df2.isnull().sum()
missing_cols = missing_before[missing_before > 0]

if len(missing_cols) == 0:
    print('  Tidak ada missing values pada dataset')
    mv_log.append('Tidak ada missing values ditemukan')
else:
    print(f'  Missing values sebelum penanganan:')
    for col, val in missing_cols.items():
        print(f'    - {col}: {val} ({val/len(df2)*100:.2f}%)')
    for col in missing_cols.index:
        if col in num_cols:
            median_val = df2[col].median()
            df2[col] = df2[col].fillna(median_val)
            mv_log.append(f'Kolom {col}: isi {missing_cols[col]} missing dengan median ({median_val:.2f})')
        else:
            mode_val = df2[col].mode()[0]
            df2[col] = df2[col].fillna(mode_val)
            mv_log.append(f'Kolom {col}: isi {missing_cols[col]} missing dengan modus ({mode_val})')

missing_after = df2.isnull().sum().sum()
print(f'\n  Missing values setelah penanganan: {missing_after}')
for log in mv_log:
    print(f'    - {log}')

In [ ]:
# ================================================================
# 1.4 REMOVE OUTLIERS (IQR METHOD)
# ================================================================
print('=' * 70)
print('TAHAP 1.4: REMOVE OUTLIERS (IQR Method)')
print('=' * 70)

df3 = df2.copy()
outlier_log = {}
outlier_rows_before = df3.shape[0]

for col in num_cols:
    Q1 = df3[col].quantile(0.25)
    Q3 = df3[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    n_outliers = ((df3[col] < lower_bound) | (df3[col] > upper_bound)).sum()
    outlier_log[col] = {'n_outliers': n_outliers,'lower_bound': round(lower_bound, 2),'upper_bound': round(upper_bound, 2),'pct': round(n_outliers / len(df3) * 100, 2)}
    df3 = df3[(df3[col] >= lower_bound) & (df3[col] <= upper_bound)]

n_total_outliers = outlier_rows_before - df3.shape[0]
print(f'  Total baris sebelum: {outlier_rows_before:,}')
print(f'  Total baris setelah:  {df3.shape[0]:,}')
print(f'  Baris terhapus:       {n_total_outliers:,} ({n_total_outliers/outlier_rows_before*100:.2f}%)')
print(f'\n  Detil outlier per kolom:')
for col, info in outlier_log.items():
    print(f'    {col}: {info["n_outliers"]:,} outliers ({info["pct"]}%) | bounds: [{info["lower_bound"]}, {info["upper_bound"]}]')

In [ ]:
# ================================================================
# 1.5 NORMALISASI & STANDARISASI FITUR
# ================================================================
print('=' * 70)
print('TAHAP 1.5: NORMALISASI & STANDARISASI FITUR')
print('=' * 70)

df4 = df3.copy()

encode_cols = cat_cols
before_encode_shape = df4.shape
df4 = pd.get_dummies(df4, columns=encode_cols, drop_first=False, dtype=int)
print(f'  One-Hot Encoding:')
print(f'    Sebelum: {before_encode_shape[1]} kolom -> Sesudah: {df4.shape[1]} kolom (+{df4.shape[1]-before_encode_shape[1]} fitur baru)')

print(f'\n  Normalisasi (Min-Max Scaling) [0,1]:')
for col in num_cols:
    col_min = df4[col].min()
    col_max = df4[col].max()
    df4[f'{col}_Normalized'] = (df4[col] - col_min) / (col_max - col_min)
    print(f'    {col}_Normalized: min={df4[f"{col}_Normalized"].min():.4f}, max={df4[f"{col}_Normalized"].max():.4f}')

print(f'\n  Standarisasi (Z-Score) [mean~0, std~1]:')
for col in num_cols:
    col_mean = df4[col].mean()
    col_std = df4[col].std()
    df4[f'{col}_Standardized'] = (df4[col] - col_mean) / col_std
    print(f'    {col}_Standardized: mean={df4[f"{col}_Standardized"].mean():.4f}, std={df4[f"{col}_Standardized"].std():.4f}')

In [ ]:
# ================================================================
# 1.6 SIMPAN HASIL PREPROCESSING
# ================================================================
output_dir_pre = 'preprocessed'
os.makedirs(output_dir_pre, exist_ok=True)

df3.to_csv(f'{output_dir_pre}/dataset_no_outlier.csv', index=False)
print(f'Dataset tanpa outlier: {output_dir_pre}/dataset_no_outlier.csv ({df3.shape[0]:,} baris)')

orig_norm_cols = [c for c in df4.columns if not c.endswith('_Standardized')]
df4_norm = df4[orig_norm_cols]
df4_norm.to_csv(f'{output_dir_pre}/normalisasi_dan_standar_fitur.csv', index=False)
print(f'Dataset normalisasi: {output_dir_pre}/normalisasi_dan_standar_fitur.csv ({df4_norm.shape[1]} kolom)')

df4.to_csv(f'{output_dir_pre}/dataset_preprocessing_lengkap.csv', index=False)
print(f'Dataset preprocessing lengkap: {output_dir_pre}/dataset_preprocessing_lengkap.csv ({df4.shape[1]} kolom)')

print(f'\n  Ringkasan Preprocessing:')
print(f'  {"Tahap":<15} {"Baris":>10} {"Kolom":>8}')
print(f'  {"-"*35}')
print(f'  {"Raw":<15} {df_raw.shape[0]:>10,} {df_raw.shape[1]:>8}')
print(f'  {"After Clean":<15} {df1.shape[0]:>10,} {df1.shape[1]:>8}')
print(f'  {"After MV":<15} {df2.shape[0]:>10,} {df2.shape[1]:>8}')
print(f'  {"No Outlier":<15} {df3.shape[0]:>10,} {df3.shape[1]:>8}')
print(f'  {"After Encode":<15} {df4.shape[0]:>10,} {df4.shape[1]:>8}')

---
## BAGIAN 2: EXPLORATORY DATA ANALYSIS (EDA)

Analisis eksploratif komprehensif dari dataset kanker kolorektal.

In [ ]:
# ============================================================
# 2.1 LOAD DATA UNTUK EDA
# ============================================================
print('=' * 70)
print('BAGIAN 2: EXPLORATORY DATA ANALYSIS')
print('=' * 70)

df_eda = pd.read_csv('colorectal_cancer_dataset_asli.csv', sep=';')
print(f'\nDataset shape: {df_eda.shape}')

df_analysis = df_eda.drop(columns=['Patient_ID'])
num_cols_eda = ['Age', 'Tumor_Size_mm', 'Healthcare_Costs', 'Incidence_Rate_per_100K', 'Mortality_Rate_per_100K']
cat_cols_eda = [c for c in df_analysis.columns if c not in num_cols_eda]

eda_out = 'eda_output'
os.makedirs(eda_out, exist_ok=True)

In [ ]:
# ============================================================
# 2.2 DATA QUALITY CHECK
# ============================================================
print('=' * 60)
print('2.2 DATA QUALITY CHECK')
print('=' * 60)

print('\n--- Missing Values ---')
missing = df_analysis.isnull().sum()
missing_pct = (missing / len(df_analysis)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Percentage (%)': missing_pct.round(2)})
print(missing_df[missing_df['Missing'] > 0] if missing_df[missing_df['Missing'] > 0].shape[0] > 0 else 'Tidak ada missing values')

print(f'\n--- Duplicated Rows ---')
print(f'Duplikat: {df_analysis.duplicated().sum()} baris')

print('\n--- Data Types ---')
print(df_analysis.dtypes.to_string())

print('\n--- Basic Info ---')
print(f'Jumlah baris: {df_analysis.shape[0]:,}')
print(f'Jumlah kolom: {df_analysis.shape[1]}')
print(f'Kolom numerik: {num_cols_eda}')
print(f'Kolom kategorikal: {len(cat_cols_eda)} kolom')

In [ ]:
# ============================================================
# 2.3 UNIVARIATE ANALYSIS - NUMERIC
# ============================================================
print('=' * 60)
print('2.3 UNIVARIATE ANALYSIS - NUMERIC VARIABLES')
print('=' * 60)

print('\n--- Numeric Summary Statistics ---')
print(df_analysis[num_cols_eda].describe().round(2).to_string())

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    ax = axes[i]
    sns.histplot(df_analysis[col], kde=True, bins=50, color='steelblue', ax=ax)
    ax.set_title(f'Distribusi {col}', fontsize=12, fontweight='bold')
    mean_val = df_analysis[col].mean()
    median_val = df_analysis[col].median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='green', linestyle=':', linewidth=1.5, label=f'Median: {median_val:.1f}')
    ax.legend(fontsize=8)

ax = axes[5]
df_box = df_analysis[num_cols_eda].melt(var_name='Variable', value_name='Value')
sns.boxplot(data=df_box, x='Variable', y='Value', palette='Set2', ax=ax)
ax.set_title('Boxplot Variabel Numerik', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{eda_out}/01_numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 2.4 UNIVARIATE ANALYSIS - CATEGORICAL
# ============================================================
print('=' * 60)
print('2.4 UNIVARIATE ANALYSIS - CATEGORICAL VARIABLES')
print('=' * 60)

key_cat_cols = ['Gender', 'Cancer_Stage', 'Country', 'Treatment_Type','Survival_5_years', 'Mortality', 'Economic_Classification','Obesity_BMI', 'Smoking_History', 'Alcohol_Consumption','Diabetes', 'Genetic_Mutation', 'Early_Detection', 'Insurance_Status']

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

cat_plot_cols = [c for c in key_cat_cols if c in df_analysis.columns][:6]
for i, col in enumerate(cat_plot_cols):
    ax = axes[i]
    value_counts = df_analysis[col].value_counts()
    colors = plt.cm.Set2(np.linspace(0, 1, len(value_counts)))
    bars = ax.barh(range(len(value_counts)), value_counts.values, color=colors)
    ax.set_yticks(range(len(value_counts)))
    ax.set_yticklabels(value_counts.index, fontsize=10)
    ax.set_title(f'Distribusi {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Jumlah')
    for bar, val in zip(bars, value_counts.values):
        ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2, f'{val:,}', ha='left', va='center', fontsize=9)

for j in range(len(cat_plot_cols), 6):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(f'{eda_out}/02_categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Frequency Tables ---')
for col in cat_cols_eda[:5]:
    print(f'\n{col}:')
    vc = df_analysis[col].value_counts()
    vc_pct = (vc / len(df_analysis) * 100).round(2)
    freq_df = pd.DataFrame({'Count': vc, 'Percentage (%)': vc_pct})
    print(freq_df.to_string())

In [ ]:
# ============================================================
# 2.5 BIVARIATE ANALYSIS - NUMERIC vs SURVIVAL
# ============================================================
print('=' * 60)
print('2.5 BIVARIATE ANALYSIS - NUMERIC vs SURVIVAL_5_YEARS')
print('=' * 60)

target = 'Survival_5_years'

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    ax = axes[i]
    for surv_val in df_analysis[target].unique():
        subset = df_analysis[df_analysis[target] == surv_val][col]
        sns.kdeplot(subset, label=f'{target}={surv_val}', ax=ax, linewidth=2)
    ax.set_title(f'{col} by {target}', fontsize=11, fontweight='bold')
    ax.legend()

ax = axes[5]
group_stats = df_analysis.groupby(target)[num_cols_eda].mean().T
group_stats.plot(kind='bar', ax=ax, colormap='Set2', width=0.8)
ax.set_title('Rata-rata Variabel Numerik by Survival', fontsize=11, fontweight='bold')
ax.set_ylabel('Rata-rata')
ax.legend(title=target)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{eda_out}/03_numeric_by_survival.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 2.6 BIVARIATE ANALYSIS - CATEGORICAL vs TARGET
# ============================================================
print('=' * 60)
print('2.6 BIVARIATE ANALYSIS - CATEGORICAL vs SURVIVAL_5_YEARS')
print('=' * 60)

cross_cols = ['Gender', 'Cancer_Stage', 'Country', 'Treatment_Type','Smoking_History', 'Alcohol_Consumption', 'Obesity_BMI','Diabetes', 'Genetic_Mutation', 'Early_Detection','Economic_Classification', 'Healthcare_Access', 'Insurance_Status']

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(cross_cols[:6]):
    ax = axes[i]
    ct = pd.crosstab(df_analysis[col], df_analysis[target], normalize='index') * 100
    ct.plot(kind='barh', stacked=True, ax=ax, colormap='RdYlGn', width=0.85)
    ax.set_title(f'{target} vs {col}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Persentase (%)')
    ax.legend(loc='lower right', title=target)

plt.tight_layout()
plt.savefig(f'{eda_out}/04_categorical_vs_survival.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Chi-Square Test ---')
chi_results = []
for col in cross_cols:
    if col in df_analysis.columns:
        ct = pd.crosstab(df_analysis[col], df_analysis[target])
        if ct.size > 0:
            chi2, p, dof, expected = chi2_contingency(ct)
            chi_results.append({'Variable': col, 'Chi2': round(chi2, 2), 'p-value': p, 'Signifikan': 'Ya' if p < 0.05 else 'Tidak'})

chi_df = pd.DataFrame(chi_results)
print(chi_df.to_string(index=False))

In [ ]:
# ============================================================
# 2.7 KORELASI MATRIKS
# ============================================================
print('=' * 60)
print('2.7 KORELASI MATRIKS')
print('=' * 60)

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df_analysis[num_cols_eda].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, mask=mask, square=True, linewidths=0.5, ax=ax, annot_kws={'fontsize': 10})
ax.set_title('Korelasi Antar Variabel Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{eda_out}/05_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n--- Correlation Values ---')
print(corr_matrix.round(4).to_string())

In [ ]:
# ============================================================
# 2.8 CANCER STAGE & TREATMENT ANALYSIS
# ============================================================
print('=' * 60)
print('2.8 CANCER STAGE & TREATMENT ANALYSIS')
print('=' * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

ax = axes[0]
ct_stage = pd.crosstab(df_analysis['Cancer_Stage'], df_analysis[target], normalize='index') * 100
ct_stage.plot(kind='bar', ax=ax, colormap='RdYlGn', width=0.8)
ax.set_title('Survival Rate by Cancer Stage', fontsize=12, fontweight='bold')
ax.set_xlabel('Cancer Stage')
ax.set_ylabel('Persentase (%)')
ax.legend(title=target)

ax = axes[1]
ct_treat = pd.crosstab(df_analysis['Treatment_Type'], df_analysis['Cancer_Stage'], normalize='index') * 100
ct_treat.plot(kind='bar', ax=ax, colormap='Set2', width=0.8)
ax.set_title('Treatment Distribution by Cancer Stage', fontsize=12, fontweight='bold')
ax.set_xlabel('Treatment Type')
ax.set_ylabel('Persentase (%)')
ax.legend(title='Cancer Stage')
ax.tick_params(axis='x', rotation=15)

ax = axes[2]
sns.boxplot(data=df_analysis, x='Cancer_Stage', y='Tumor_Size_mm', palette='Set2', ax=ax)
ax.set_title('Tumor Size by Cancer Stage', fontsize=12, fontweight='bold')
ax.set_xlabel('Cancer Stage')
ax.set_ylabel('Tumor Size (mm)')

plt.tight_layout()
plt.savefig(f'{eda_out}/06_stage_treatment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 2.9 RISK FACTORS ANALYSIS
# ============================================================
print('=' * 60)
print('2.9 RISK FACTORS & DEMOGRAPHIC ANALYSIS')
print('=' * 60)

risk_factors = ['Smoking_History', 'Alcohol_Consumption', 'Obesity_BMI','Diabetes', 'Genetic_Mutation', 'Family_History','Diet_Risk', 'Physical_Activity', 'Inflammatory_Bowel_Disease']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, factor in enumerate(risk_factors[:6]):
    if factor in df_analysis.columns:
        ax = axes[i]
        ct = pd.crosstab(df_analysis[factor], df_analysis[target], normalize='index') * 100
        ct.plot(kind='barh', stacked=True, ax=ax, colormap='RdYlGn', width=0.8)
        ax.set_title(f'{target} by {factor}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Persentase (%)')
        ax.legend(loc='lower right', title=target, fontsize=8)

plt.tight_layout()
plt.savefig(f'{eda_out}/07_risk_factors_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 2.10 SURVIVAL PREDICTION ANALYSIS
# ============================================================
print('=' * 60)
print('2.10 SURVIVAL PREDICTION TARGET ANALYSIS')
print('=' * 60)

print('\n--- Survival Prediction Distribution ---')
print(df_analysis['Survival_Prediction'].value_counts())

print('\n--- Survival Prediction vs Actual Survival ---')
ct_pred = pd.crosstab(df_analysis['Survival_Prediction'], df_analysis['Survival_5_years'])
print(ct_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ct_pred_pct = pd.crosstab(df_analysis['Survival_Prediction'], df_analysis[target], normalize='index') * 100
ct_pred_pct.plot(kind='barh', stacked=True, ax=ax, colormap='RdYlGn', width=0.8)
ax.set_title('Actual Survival by Prediction', fontsize=12, fontweight='bold')
ax.set_xlabel('Persentase (%)')
ax.legend(title=target)

ax = axes[1]
df_analysis['Survival_Prediction'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=plt.cm.Set3(np.linspace(0, 1, 5)), ax=ax, explode=[0.05]*5)
ax.set_title('Survival Prediction Distribution', fontsize=12, fontweight='bold')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig(f'{eda_out}/08_survival_prediction_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nEDA SELESAI! Semua output tersimpan di folder: {eda_out}/')

---
## BAGIAN 3: TARGET ANALYSIS

Membandingkan semua target yang mungkin untuk menentukan mana yang paling prediktif.

In [ ]:
# ================================================================
# 3.1 LOAD DATA
# ================================================================
print('=' * 80)
print('BAGIAN 3: TARGET ANALYSIS - Cari Target Paling Prediktif')
print('=' * 80)

df_target = pd.read_csv('preprocessed/dataset_no_outlier.csv')
print(f'Dataset: {df_target.shape[0]:,} baris, {df_target.shape[1]} kolom')

df_target_sample = df_target.sample(n=50000, random_state=42)
print(f'Sample: {df_target_sample.shape[0]:,} baris')

drop_cols_target = ['Patient_ID', 'Survival_Prediction']
df_ml_target = df_target_sample.drop(columns=[c for c in drop_cols_target if c in df_target_sample.columns])

In [ ]:
# ================================================================
# 3.2 DEFINE & EVALUATE TARGET CANDIDATES
# ================================================================
print('\n3.2 TARGET CANDIDATES')

target_candidates = {
    'Survival_5_years': {'type': 'classification', 'desc': 'Survival 5 tahun (Yes/No)', 'is_binary': True},
    'Cancer_Stage': {'type': 'classification', 'desc': 'Stage kanker (multi-class)', 'is_binary': False},
    'Early_Detection': {'type': 'classification', 'desc': 'Deteksi dini (Yes/No)', 'is_binary': True},
    'Treatment_Type': {'type': 'classification', 'desc': 'Jenis treatment (multi-class)', 'is_binary': False},
    'Mortality': {'type': 'classification', 'desc': 'Mortalitas (Yes/No)', 'is_binary': True},
    'Healthcare_Costs': {'type': 'regression', 'desc': 'Biaya healthcare (numeric)', 'is_binary': False},
    'Tumor_Size_mm': {'type': 'regression', 'desc': 'Ukuran tumor (numeric)', 'is_binary': False}
}

results_list = []
num_cols_target = ['Age', 'Tumor_Size_mm', 'Healthcare_Costs', 'Incidence_Rate_per_100K', 'Mortality_Rate_per_100K']

for target_name, target_info in target_candidates.items():
    if target_name not in df_ml_target.columns:
        print(f'  [!] {target_name} tidak ditemukan, skip')
        continue
    print(f'\n  Target: {target_name} ({target_info["desc"]})')
    other_targets = [t for t in target_candidates.keys() if t != target_name and t in df_ml_target.columns]
    feature_cols = [c for c in df_ml_target.columns if c not in other_targets and c != target_name]
    num_c = [c for c in num_cols_target if c in feature_cols]
    cat_c = [c for c in feature_cols if c not in num_c]
    df_encoded = pd.get_dummies(df_ml_target[feature_cols], columns=cat_c, drop_first=True, dtype=int)
    X = df_encoded.values
    y_raw = df_ml_target[target_name].values
    if target_info['type'] == 'classification':
        le = LabelEncoder()
        y = le.fit_transform(y_raw)
        print(f'  Kelas ({len(le.classes_)}): {list(le.classes_)}')
    else:
        y = y_raw.astype(float)
        print(f'  Range: [{y.min():.2f}, {y.max():.2f}], Mean: {y.mean():.2f}')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    print(f'  Train: {X_train.shape[0]:,}, Test: {X_test.shape[0]:,}')
    if target_info['type'] == 'classification':
        start = time.time()
        rf = RF(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
        rf.fit(X_train_s, y_train)
        elapsed = time.time() - start
        y_pred = rf.predict(X_test_s)
        acc = accuracy_score(y_test, y_pred)
        dummy = DummyClassifier(strategy='most_frequent')
        dummy.fit(X_train_s, y_train)
        dummy_acc = accuracy_score(y_test, dummy.predict(X_test_s))
        improvement = ((acc - dummy_acc) / dummy_acc) * 100
        if target_info['is_binary'] and len(np.unique(y)) == 2:
            y_prob = rf.predict_proba(X_test_s)[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        else:
            try:
                y_prob = rf.predict_proba(X_test_s)
                auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
            except:
                auc = 0.0
        result = {'Target': target_name, 'Type': target_info['type'], 'Accuracy': round(acc, 4), 'Baseline': round(dummy_acc, 4), 'Improvement (%)': round(improvement, 2), 'ROC-AUC': round(auc, 4), 'Time (s)': round(elapsed, 2)}
        print(f'  Accuracy: {acc:.4f} (Baseline: {dummy_acc:.4f}, +{improvement:.1f}%)')
        print(f'  ROC-AUC: {auc:.4f}')
    else:
        start = time.time()
        rf = RF(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
        rf.fit(X_train_s, y_train)
        elapsed = time.time() - start
        y_pred = rf.predict(X_test_s)
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        result = {'Target': target_name, 'Type': target_info['type'], 'R2 Score': round(r2, 4), 'RMSE': round(rmse, 2), 'Time (s)': round(elapsed, 2)}
        print(f'  R2 Score: {r2:.4f}, RMSE: {rmse:.2f}')
    results_list.append(result)

results_df_target = pd.DataFrame(results_list)
print('\n' + '=' * 80)
print('3.3 HASIL PERBANDINGAN TARGET')
print('=' * 80)
for _, row in results_df_target.iterrows():
    if row['Type'] == 'classification':
        print(f'  {row["Target"]:<25} Acc: {row["Accuracy"]:.4f} | AUC: {row["ROC-AUC"]:.4f} | Improv: {row.get("Improvement (%)", 0):.1f}%')
    else:
        print(f'  {row["Target"]:<25} R2: {row.get("R2 Score", 0):.4f} | RMSE: {row.get("RMSE", 0):.2f}')

---
## BAGIAN 4: TRAINING IMPORTANCE FITUR

Feature Importance untuk target **Survival_Prediction** + training 4 model dengan SMOTE K=5.

- Feature Importance (RF + Mutual Information)
- Seleksi fitur terbaik
- SMOTE K=5 untuk imbalance data
- 4 Algoritma: SVM, XGBoost, Random Forest, KNN + Baseline
- Confusion Matrix, ROC Curves, Prediction Table

In [ ]:
# ================================================================
# 4.1 LOAD & PREPARE DATA
# ================================================================
print('=' * 80)
print('BAGIAN 4: TRAINING IMPORTANCE FITUR - SURVIVAL_PREDICTION')
print('=' * 80)

output_dir_fi = 'Training_Importance_Fitur'
os.makedirs(output_dir_fi, exist_ok=True)

df_fi = pd.read_csv('preprocessed/dataset_no_outlier.csv')
print(f'Total dataset: {df_fi.shape[0]:,} baris, {df_fi.shape[1]} kolom')

df_fi['Survival_Prediction'] = df_fi['Survival_Prediction'].map({'Yes': 1, 'No': 0})

df_fi_sample, _ = train_test_split(df_fi, train_size=30000, random_state=42, stratify=df_fi['Survival_Prediction'])
print(f'Sample: {df_fi_sample.shape[0]:,} baris (stratified)')
print(f'  Yes=1: {df_fi_sample["Survival_Prediction"].sum():,} ({df_fi_sample["Survival_Prediction"].mean()*100:.1f}%)')
print(f'  No=0:  {(df_fi_sample["Survival_Prediction"]==0).sum():,} ({(1-df_fi_sample["Survival_Prediction"].mean())*100:.1f}%)')

drop_cols_fi = ['Patient_ID', 'Survival_5_years', 'Mortality']
df_ml_fi = df_fi_sample.drop(columns=[c for c in drop_cols_fi if c in df_fi_sample.columns])

num_cols_fi = ['Age', 'Tumor_Size_mm', 'Healthcare_Costs', 'Incidence_Rate_per_100K', 'Mortality_Rate_per_100K']
cat_cols_fi = [c for c in df_ml_fi.columns if c not in num_cols_fi and c != 'Survival_Prediction']
print(f'\nFitur numerik ({len(num_cols_fi)}): {num_cols_fi}')
print(f'Fitur kategorikal ({len(cat_cols_fi)}): {cat_cols_fi}')

df_encoded_fi = pd.get_dummies(df_ml_fi, columns=cat_cols_fi, drop_first=True, dtype=int)
feature_names_fi = [c for c in df_encoded_fi.columns if c != 'Survival_Prediction']
print(f'Total fitur setelah encoding: {len(feature_names_fi)}')

X_fi = df_encoded_fi[feature_names_fi].values
y_fi = df_encoded_fi['Survival_Prediction'].values

del df_fi, df_fi_sample, df_ml_fi, df_encoded_fi
gc.collect()

In [ ]:
# ================================================================
# 4.2 FEATURE IMPORTANCE (Random Forest)
# ================================================================
print('=' * 80)
print('4.2 FEATURE IMPORTANCE - RANDOM FOREST')
print('=' * 80)

rf_importance = RF(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_importance.fit(X_fi, y_fi)

fi_scores = rf_importance.feature_importances_
fi_df = pd.DataFrame({'Feature': feature_names_fi, 'Importance': fi_scores})
fi_df = fi_df.sort_values('Importance', ascending=False).reset_index(drop=True)

print('\n  Top 15 fitur terpenting untuk Survival_Prediction:')
print(f'  {"No":>3} {"Feature":<55} {"Importance":>10}')
print(f'  {"-"*70}')
for i in range(min(15, len(fi_df))):
    print(f'  {i+1:>3}. {fi_df.loc[i, "Feature"]:<55} {fi_df.loc[i, "Importance"]:.6f}')

In [ ]:
# ================================================================
# 4.3 MUTUAL INFORMATION SCORE
# ================================================================
print('=' * 80)
print('4.3 MUTUAL INFORMATION SCORE')
print('=' * 80)

mi_selector = SelectKBest(mutual_info_classif, k='all')
mi_scores = mi_selector.fit(X_fi, y_fi)
mi_df = pd.DataFrame({'Feature': feature_names_fi, 'MI_Score': mi_scores.scores_})
mi_df = mi_df.sort_values('MI_Score', ascending=False).reset_index(drop=True)

print('\n  Top 15 fitur (Mutual Information):')
for i in range(min(15, len(mi_df))):
    print(f'  {i+1:>3}. {mi_df.loc[i, "Feature"]:<55} {mi_df.loc[i, "MI_Score"]:.6f}')

In [ ]:
# ================================================================
# 4.4 VISUALISASI FEATURE IMPORTANCE
# ================================================================
print('=' * 80)
print('4.4 VISUALISASI FEATURE IMPORTANCE')
print('=' * 80)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
top_n = 15

ax = axes[0]
fi_top = fi_df.head(top_n)
colors = plt.cm.Greens(np.linspace(0.3, 0.9, top_n))[::-1]
ax.barh(range(top_n), fi_top['Importance'].values, color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels(fi_top['Feature'].values, fontsize=8)
ax.set_xlabel('Importance Score')
ax.set_title(f'Top {top_n} Features - Random Forest', fontsize=13, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(fi_top['Importance'].values):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

ax = axes[1]
mi_top = mi_df.head(top_n)
colors2 = plt.cm.Blues(np.linspace(0.3, 0.9, top_n))[::-1]
ax.barh(range(top_n), mi_top['MI_Score'].values, color=colors2)
ax.set_yticks(range(top_n))
ax.set_yticklabels(mi_top['Feature'].values, fontsize=8)
ax.set_xlabel('Mutual Information Score')
ax.set_title(f'Top {top_n} Features - Mutual Information', fontsize=13, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(mi_top['MI_Score'].values):
    ax.text(v + 0.0001, i, f'{v:.4f}', va='center', fontsize=8)

plt.suptitle('FEATURE IMPORTANCE untuk TARGET Survival_Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir_fi}/01_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

fi_df.to_csv(f'{output_dir_fi}/feature_importance_rf.csv', index=False)
mi_df.to_csv(f'{output_dir_fi}/feature_importance_mutual_info.csv', index=False)
print('  => Saved: feature_importance_rf.csv & feature_importance_mutual_info.csv')

In [ ]:
# ================================================================
# 4.5 SELEKSI FITUR TERBAIK
# ================================================================
print('=' * 80)
print('4.5 SELEKSI FITUR TERBAIK')
print('=' * 80)

top_rf = set(fi_df.head(20)['Feature'])
top_mi = set(mi_df.head(20)['Feature'])
selected_features = list(top_rf & top_mi)
if len(selected_features) < 10:
    extra = [f for f in fi_df.head(15)['Feature'] if f not in selected_features]
    selected_features.extend(extra[:min(10-len(selected_features), len(extra))])
selected_features.sort()

print(f'  Top 20 RF:        {len(top_rf)} fitur')
print(f'  Top 20 MI:        {len(top_mi)} fitur')
print(f'  Intersection:     {len(top_rf & top_mi)} fitur')
print(f'  Selected:         {len(selected_features)} fitur')

selected_indices = [feature_names_fi.index(f) for f in selected_features]
X_selected = X_fi[:, selected_indices]

print(f'\n  Fitur terpilih ({len(selected_features)}):')
for i, f in enumerate(selected_features):
    print(f'    {i+1:>3}. {f}')

pd.DataFrame({'Feature': selected_features}).to_csv(f'{output_dir_fi}/selected_features.csv', index=False)
print('  => Saved: selected_features.csv')

In [ ]:
# ================================================================
# 4.6 SPLIT DATA + SMOTE K=5
# ================================================================
print('=' * 80)
print('4.6 SPLIT DATA + SMOTE OVERSAMPLING K=5')
print('=' * 80)

X_train_fi, X_test_fi, y_train_fi, y_test_fi = train_test_split(X_selected, y_fi, test_size=0.2, random_state=42, stratify=y_fi)

print(f'\n  SEBELUM SMOTE:')
print(f'    Train: {X_train_fi.shape[0]:,} baris (Yes={y_train_fi.sum():,}, No={(y_train_fi==0).sum():,})')
print(f'    Ratio: {y_train_fi.mean()*100:.1f}% / {(1-y_train_fi.mean())*100:.1f}%')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_fi, y_train_fi)

print(f'\n  SETELAH SMOTE (K=5):')
print(f'    Train: {X_train_res.shape[0]:,} baris (+{X_train_res.shape[0]-X_train_fi.shape[0]:,})')
print(f'    Yes={y_train_res.sum():,}, No={(y_train_res==0).sum():,}')
print(f'    Ratio: {y_train_res.mean()*100:.1f}% / {(1-y_train_res.mean())*100:.1f}%')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test_fi)

In [ ]:
# ================================================================
# 4.7 TRAINING 4 MODEL + BASELINE
# ================================================================
print('=' * 80)
print('4.7 TRAINING 4 ALGORITMA + BASELINE')
print('=' * 80)

models_fi = {
    'SVM': CalibratedClassifierCV(LinearSVC(C=1.0, class_weight='balanced', max_iter=2000, random_state=42, dual='auto'), cv=3, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss'),
    'Random Forest': RF(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1),
    'KNN': KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),
    'Baseline': DummyClassifier(strategy='most_frequent')
}

results_fi = {}
predictions_fi = {}
probabilities_fi = {}
model_names_fi = list(models_fi.keys())

for name, model in models_fi.items():
    print(f'\n  [{name}] Training...')
    X_tr = X_train_scaled if name in ['SVM', 'KNN'] else X_train_res
    X_te = X_test_scaled if name in ['SVM', 'KNN'] else X_test_fi
    start = time.time()
    model.fit(X_tr, y_train_res)
    elapsed = time.time() - start
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    predictions_fi[name] = y_pred
    probabilities_fi[name] = y_prob
    acc = accuracy_score(y_test_fi, y_pred)
    prec = precision_score(y_test_fi, y_pred, zero_division=0)
    rec = recall_score(y_test_fi, y_pred, zero_division=0)
    f1 = f1_score(y_test_fi, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test_fi, y_prob)
    results_fi[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1, 'ROC-AUC': roc_auc, 'Training_Time': elapsed}
    cm = confusion_matrix(y_test_fi, y_pred)
    print(f'    Accuracy:  {acc:.4f}')
    print(f'    Precision: {prec:.4f}')
    print(f'    Recall:    {rec:.4f}')
    print(f'    F1-Score:  {f1:.4f}')
    print(f'    ROC-AUC:   {roc_auc:.4f}')
    print(f'    Time:      {elapsed:.2f}s')

In [ ]:
# ================================================================
# 4.8 PREDICTION TABLE
# ================================================================
print('=' * 80)
print('4.8 PREDICTION TABLE (Actual vs Predicted - 50 samples)')
print('=' * 80)

n_show = 50
pred_table_data = []
for i in range(min(n_show, len(y_test_fi))):
    row_data = {'Actual': 'Yes' if y_test_fi[i] == 1 else 'No'}
    for name in model_names_fi:
        row_data[f'{name}'] = 'Yes' if predictions_fi[name][i] == 1 else 'No'
    pred_table_data.append(row_data)

pred_df = pd.DataFrame(pred_table_data)
print(pred_df.head(10).to_string(index=False))
pred_df.to_csv(f'{output_dir_fi}/prediction_table.csv', index=False)
print(f'\n  => Saved: prediction_table.csv ({len(pred_table_data)} samples)')

In [ ]:
# ================================================================
# 4.9 CONFUSION MATRICES
# ================================================================
print('=' * 80)
print('4.9 CONFUSION MATRICES')
print('=' * 80)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
cmaps = ['Blues', 'Greens', 'Oranges', 'Purples', 'Reds']

for i, name in enumerate(model_names_fi):
    ax = axes[i]
    cm = confusion_matrix(y_test_fi, predictions_fi[name])
    cm_pct = cm / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm, annot=True, fmt=',', cmap=cmaps[i], ax=ax, cbar=True, square=True, xticklabels=['Pred No', 'Pred Yes'], yticklabels=['Act No', 'Act Yes'], linewidths=1, linecolor='white', annot_kws={'fontsize': 14, 'fontweight': 'bold'})
    for j in range(2):
        for k in range(2):
            color = 'white' if cm_pct[j, k] > 50 else 'black'
            ax.text(k + 0.5, j + 0.65, f'({cm_pct[j, k]:.1f}%)', ha='center', va='center', fontsize=10, fontweight='bold', color=color)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')

for j in range(len(model_names_fi), 6):
    axes[j].set_visible(False)
plt.suptitle('CONFUSION MATRICES - Survival_Prediction (SMOTE K=5)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir_fi}/04_all_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# 4.10 ROC CURVES
# ================================================================
print('=' * 80)
print('4.10 ROC CURVES')
print('=' * 80)

roc_models = [n for n in model_names_fi if n != 'Baseline']

fig, ax = plt.subplots(figsize=(10, 8))
colors_roc = {'SVM': 'blue', 'XGBoost': 'green', 'Random Forest': 'orange', 'KNN': 'purple'}
linestyles = {'SVM': '-', 'XGBoost': '--', 'Random Forest': '-.', 'KNN': ':'}

for name in roc_models:
    y_score = probabilities_fi[name]
    if y_score is not None and len(np.unique(y_score)) > 1:
        fpr, tpr, _ = roc_curve(y_test_fi, y_score)
        auc_val = results_fi[name]['ROC-AUC']
        ax.plot(fpr, tpr, linestyles[name], color=colors_roc[name], linewidth=2.5, label=f'{name} (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3, label='Random (AUC = 0.5)')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC CURVES - Survival_Prediction (SMOTE K=5)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{output_dir_fi}/05_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================================================
# 4.11 PERFORMANCE COMPARISON TABLE
# ================================================================
print('=' * 80)
print('4.11 PERFORMANCE COMPARISON')
print('=' * 80)

results_df_fi = pd.DataFrame(results_fi).T.round(4)
results_df_fi['Rank_Acc'] = results_df_fi['Accuracy'].rank(ascending=False).astype(int)
results_df_fi['Rank_F1'] = results_df_fi['F1-Score'].rank(ascending=False).astype(int)
results_df_fi['Rank_AUC'] = results_df_fi['ROC-AUC'].rank(ascending=False).astype(int)
results_df_fi['Avg_Rank'] = results_df_fi[['Rank_Acc', 'Rank_F1', 'Rank_AUC']].mean(axis=1).round(2)
results_df_fi = results_df_fi.sort_values('Avg_Rank')

print(f'\n{"Algoritma":<16} {"Accuracy":>9} {"Precision":>9} {"Recall":>9} {"F1":>9} {"AUC":>9} {"Time":>8} {"Rank":>5}')
print('-' * 75)
for idx, row in results_df_fi.iterrows():
    print(f'{idx:<16} {row["Accuracy"]:>9.4f} {row["Precision"]:>9.4f} {row["Recall"]:>9.4f} {row["F1-Score"]:>9.4f} {row["ROC-AUC"]:>9.4f} {row["Training_Time"]:>7.2f}s {row["Avg_Rank"]:>4.1f}')
print('-' * 75)

results_df_fi.to_csv(f'{output_dir_fi}/model_comparison_results.csv')
print(f'\n  => Saved: model_comparison_results.csv')

print(f'\nBEST METRICS:')
print(f'  Accuracy:  {results_df_fi["Accuracy"].max():.4f} ({results_df_fi["Accuracy"].idxmax()})')
print(f'  F1-Score:  {results_df_fi["F1-Score"].max():.4f} ({results_df_fi["F1-Score"].idxmax()})')
print(f'  ROC-AUC:   {results_df_fi["ROC-AUC"].max():.4f} ({results_df_fi["ROC-AUC"].idxmax()})')

---
## BAGIAN 5: ADVANCED SOLUTIONS

Mencoba 6 pendekatan lanjutan:
- Neural Network (MLPClassifier)
- Polynomial Features + Random Forest
- PCA + XGBoost
- Stacking Ensemble
- GridSearch XGBoost
- Baseline (DummyClassifier)

In [ ]:
# ================================================================
# 5.1 LOAD DATA (5K sample)
# ================================================================
print('=' * 80)
print('BAGIAN 5: SOLUSI LANJUTAN - PREDIKSI SURVIVAL_PREDICTION')
print('=' * 80)

df_adv = pd.read_csv('preprocessed/dataset_no_outlier.csv')
df_adv['Survival_Prediction'] = df_adv['Survival_Prediction'].map({'Yes': 1, 'No': 0})
print(f'Total dataset: {df_adv.shape[0]:,} baris')

df_sample_adv, _ = train_test_split(df_adv, train_size=5000, random_state=42, stratify=df_adv['Survival_Prediction'])
y_sample_adv = df_sample_adv['Survival_Prediction'].values
print(f'Sample: 5,000 baris (Yes={y_sample_adv.sum():,}, No={(y_sample_adv==0).sum():,})')

In [ ]:
# ================================================================
# 5.2 FEATURE ENGINEERING
# ================================================================
print('=' * 80)
print('5.2 FEATURE ENGINEERING')
print('=' * 80)

num_cols_adv = ['Age', 'Tumor_Size_mm', 'Healthcare_Costs', 'Incidence_Rate_per_100K', 'Mortality_Rate_per_100K']
cat_cols_adv = ['Country', 'Gender', 'Cancer_Stage', 'Family_History', 'Smoking_History', 'Alcohol_Consumption', 'Obesity_BMI', 'Diet_Risk', 'Physical_Activity', 'Diabetes', 'Inflammatory_Bowel_Disease', 'Genetic_Mutation', 'Screening_History', 'Early_Detection', 'Treatment_Type', 'Urban_or_Rural', 'Economic_Classification', 'Healthcare_Access', 'Insurance_Status']

df_fe = df_sample_adv.copy()

risk_factors_adv = ['Smoking_History', 'Alcohol_Consumption', 'Obesity_BMI', 'Diabetes', 'Genetic_Mutation', 'Family_History', 'Inflammatory_Bowel_Disease']
for col in risk_factors_adv:
    df_fe[col + '_bin'] = (df_fe[col] == 'Yes').astype(int)
df_fe['Risk_Score'] = df_fe[[c+'_bin' for c in risk_factors_adv]].sum(axis=1)
print(f'  Created: Risk_Score')

df_fe['PA_bin'] = (df_fe['Physical_Activity'] == 'High').astype(int)
df_fe['Diet_bin'] = (df_fe['Diet_Risk'] == 'Low').astype(int)
df_fe['Screen_bin'] = (df_fe['Screening_History'] == 'Regular').astype(int)
df_fe['Healthy_Score'] = df_fe[['PA_bin', 'Diet_bin', 'Screen_bin']].sum(axis=1)
print('  Created: Healthy_Score')

df_fe['Tumor_Age_Ratio'] = df_fe['Tumor_Size_mm'] / (df_fe['Age'] + 1)
df_fe['Cost_Age_Ratio'] = df_fe['Healthcare_Costs'] / (df_fe['Age'] + 1)
df_fe['Cost_Tumor_Ratio'] = df_fe['Healthcare_Costs'] / (df_fe['Tumor_Size_mm'] + 1)
df_fe['Mortality_Incidence_Ratio'] = df_fe['Mortality_Rate_per_100K'] / (df_fe['Incidence_Rate_per_100K'] + 1)
print('  Created: 4 Ratio features')

eng_features = ['Risk_Score', 'Healthy_Score', 'Tumor_Age_Ratio', 'Cost_Age_Ratio', 'Cost_Tumor_Ratio', 'Mortality_Incidence_Ratio']
all_feats = num_cols_adv + eng_features + cat_cols_adv

df_enc_adv = pd.get_dummies(df_fe[all_feats + ['Survival_Prediction']], columns=cat_cols_adv, drop_first=True, dtype=int)

feature_names_adv = [c for c in df_enc_adv.columns if c != 'Survival_Prediction']
X_adv = df_enc_adv[feature_names_adv].values
y_adv = df_enc_adv['Survival_Prediction'].values
print(f'Total fitur: {len(feature_names_adv)}')

del df_adv, df_sample_adv, df_fe, df_enc_adv
gc.collect()

In [ ]:
# ================================================================
# 5.3 SPLIT + SMOTE
# ================================================================
print('=' * 80)
print('5.3 SPLIT + SMOTE')
print('=' * 80)

X_train_adv, X_test_adv, y_train_adv, y_test_adv = train_test_split(X_adv, y_adv, test_size=0.2, random_state=42, stratify=y_adv)

smote_adv = SMOTE(random_state=42, k_neighbors=5)
X_train_res_adv, y_train_res_adv = smote_adv.fit_resample(X_train_adv, y_train_adv)
print(f'Train: {X_train_adv.shape[0]:,} -> {X_train_res_adv.shape[0]:,} (after SMOTE)')
print(f'Test:  {X_test_adv.shape[0]:,}')

scaler_adv = StandardScaler()
X_train_scaled_adv = scaler_adv.fit_transform(X_train_res_adv)
X_test_scaled_adv = scaler_adv.transform(X_test_adv)
gc.collect()

results_adv = {}

In [ ]:
# ================================================================
# 5.4 BASELINE
# ================================================================
print('=' * 80)
print('5.4 BASELINE')
print('=' * 80)

dummy_adv = DummyClassifier(strategy='most_frequent')
dummy_adv.fit(X_train_scaled_adv, y_train_res_adv)
y_pred_d = dummy_adv.predict(X_test_scaled_adv)
y_prob_d = dummy_adv.predict_proba(X_test_scaled_adv)[:, 1]
results_adv['Baseline'] = {'Accuracy': accuracy_score(y_test_adv, y_pred_d), 'Precision': precision_score(y_test_adv, y_pred_d, zero_division=0), 'Recall': recall_score(y_test_adv, y_pred_d, zero_division=0), 'F1-Score': f1_score(y_test_adv, y_pred_d, zero_division=0), 'ROC-AUC': roc_auc_score(y_test_adv, y_prob_d), 'Time': 0}
print(f'  Baseline: Acc={results_adv["Baseline"]["Accuracy"]:.4f}, AUC={results_adv["Baseline"]["ROC-AUC"]:.4f}')

In [ ]:
# ================================================================
# 5.5 NEURAL NETWORK
# ================================================================
print('=' * 80)
print('5.5 NEURAL NETWORK (MLPClassifier)')
print('=' * 80)

mlp = MLPClassifier(hidden_layer_sizes=(64, 32, 16), activation='relu', max_iter=300, random_state=42, early_stopping=True)
start = time.time()
mlp.fit(X_train_scaled_adv, y_train_res_adv)
elapsed = time.time() - start
y_pred = mlp.predict(X_test_scaled_adv)
y_prob = mlp.predict_proba(X_test_scaled_adv)[:, 1]
results_adv['Neural Network'] = {'Accuracy': accuracy_score(y_test_adv, y_pred), 'Precision': precision_score(y_test_adv, y_pred, zero_division=0), 'Recall': recall_score(y_test_adv, y_pred, zero_division=0), 'F1-Score': f1_score(y_test_adv, y_pred, zero_division=0), 'ROC-AUC': roc_auc_score(y_test_adv, y_prob), 'Time': round(elapsed, 2)}
print(f'  Acc={results_adv["Neural Network"]["Accuracy"]:.4f}, AUC={results_adv["Neural Network"]["ROC-AUC"]:.4f} ({elapsed:.1f}s)')

In [ ]:
# ================================================================
# 5.6 POLYNOMIAL FEATURES + RF
# ================================================================
print('=' * 80)
print('5.6 POLYNOMIAL FEATURES (degree=2, interaction_only) + RF')
print('=' * 80)

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled_adv)
X_test_poly = poly.transform(X_test_scaled_adv)
print(f'  Fitur: {X_train_scaled_adv.shape[1]} -> {X_train_poly.shape[1]}')

rf_poly = RF(n_estimators=50, max_depth=6, random_state=42, n_jobs=1)
start = time.time()
rf_poly.fit(X_train_poly, y_train_res_adv)
elapsed = time.time() - start
y_pred = rf_poly.predict(X_test_poly)
y_prob = rf_poly.predict_proba(X_test_poly)[:, 1]
results_adv['RF + Polynomial'] = {'Accuracy': accuracy_score(y_test_adv, y_pred), 'Precision': precision_score(y_test_adv, y_pred, zero_division=0), 'Recall': recall_score(y_test_adv, y_pred, zero_division=0), 'F1-Score': f1_score(y_test_adv, y_pred, zero_division=0), 'ROC-AUC': roc_auc_score(y_test_adv, y_prob), 'Time': round(elapsed, 2)}
print(f'  Acc={results_adv["RF + Polynomial"]["Accuracy"]:.4f}, AUC={results_adv["RF + Polynomial"]["ROC-AUC"]:.4f} ({elapsed:.1f}s)')
del X_train_poly, X_test_poly, rf_poly
gc.collect()

In [ ]:
# ================================================================
# 5.7 PCA + XGBoost
# ================================================================
print('=' * 80)
print('5.7 PCA + XGBoost')
print('=' * 80)

pca = PCA(n_components=0.90)
X_train_pca = pca.fit_transform(X_train_scaled_adv)
X_test_pca = pca.transform(X_test_scaled_adv)
print(f'  Fitur: {X_train_scaled_adv.shape[1]} -> {X_train_pca.shape[1]} (90% variance)')

xgb_pca = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, n_jobs=1, eval_metric='logloss')
start = time.time()
xgb_pca.fit(X_train_pca, y_train_res_adv)
elapsed = time.time() - start
y_pred = xgb_pca.predict(X_test_pca)
y_prob = xgb_pca.predict_proba(X_test_pca)[:, 1]
results_adv['XGBoost + PCA'] = {'Accuracy': accuracy_score(y_test_adv, y_pred), 'Precision': precision_score(y_test_adv, y_pred, zero_division=0), 'Recall': recall_score(y_test_adv, y_pred, zero_division=0), 'F1-Score': f1_score(y_test_adv, y_pred, zero_division=0), 'ROC-AUC': roc_auc_score(y_test_adv, y_prob), 'Time': round(elapsed, 2)}
print(f'  Acc={results_adv["XGBoost + PCA"]["Accuracy"]:.4f}, AUC={results_adv["XGBoost + PCA"]["ROC-AUC"]:.4f} ({elapsed:.1f}s)')
del X_train_pca, X_test_pca, xgb_pca
gc.collect()

In [ ]:
# ================================================================
# 5.8 STACKING ENSEMBLE
# ================================================================
print('=' * 80)
print('5.8 STACKING ENSEMBLE')
print('=' * 80)

estimators = [
    ('svm', CalibratedClassifierCV(LinearSVC(C=1.0, class_weight='balanced', max_iter=2000, random_state=42, dual='auto'), cv=3, n_jobs=1)),
    ('xgb', XGBClassifier(n_estimators=50, max_depth=4, random_state=42, n_jobs=1, eval_metric='logloss')),
    ('rf', RF(n_estimators=30, max_depth=6, random_state=42, n_jobs=1))
]
stack = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), cv=3, n_jobs=1)
start = time.time()
stack.fit(X_train_scaled_adv, y_train_res_adv)
elapsed = time.time() - start
y_pred = stack.predict(X_test_scaled_adv)
y_prob = stack.predict_proba(X_test_scaled_adv)[:, 1]
results_adv['Stacking Ensemble'] = {'Accuracy': accuracy_score(y_test_adv, y_pred), 'Precision': precision_score(y_test_adv, y_pred, zero_division=0), 'Recall': recall_score(y_test_adv, y_pred, zero_division=0), 'F1-Score': f1_score(y_test_adv, y_pred, zero_division=0), 'ROC-AUC': roc_auc_score(y_test_adv, y_prob), 'Time': round(elapsed, 2)}
print(f'  Acc={results_adv["Stacking Ensemble"]["Accuracy"]:.4f}, AUC={results_adv["Stacking Ensemble"]["ROC-AUC"]:.4f} ({elapsed:.1f}s)')
del stack
gc.collect()

In [ ]:
# ================================================================
# 5.9 GRIDSEARCH XGBoost
# ================================================================
print('=' * 80)
print('5.9 GRIDSEARCH XGBoost')
print('=' * 80)

param_grid = {'n_estimators': [30, 50], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(XGBClassifier(random_state=42, n_jobs=1, eval_metric='logloss'), param_grid, cv=cv, scoring='roc_auc', n_jobs=1, verbose=0)
start = time.time()
grid.fit(X_train_scaled_adv, y_train_res_adv)
elapsed = time.time() - start
y_pred = grid.predict(X_test_scaled_adv)
y_prob = grid.predict_proba(X_test_scaled_adv)[:, 1]
results_adv['XGBoost GridSearch'] = {'Accuracy': accuracy_score(y_test_adv, y_pred), 'Precision': precision_score(y_test_adv, y_pred, zero_division=0), 'Recall': recall_score(y_test_adv, y_pred, zero_division=0), 'F1-Score': f1_score(y_test_adv, y_pred, zero_division=0), 'ROC-AUC': roc_auc_score(y_test_adv, y_prob), 'Best Params': str(grid.best_params_), 'Time': round(elapsed, 2)}
print(f'  Best params: {grid.best_params_}')
print(f'  Acc={results_adv["XGBoost GridSearch"]["Accuracy"]:.4f}, AUC={results_adv["XGBoost GridSearch"]["ROC-AUC"]:.4f} ({elapsed:.1f}s)')

In [ ]:
# ================================================================
# 5.10 PERBANDINGAN & VISUALISASI
# ================================================================
print('=' * 80)
print('5.10 PERBANDINGAN SEMUA SOLUSI')
print('=' * 80)

results_df_adv = pd.DataFrame(results_adv).T.round(4)
results_df_adv = results_df_adv.sort_values('ROC-AUC', ascending=False)

print(f'\n{"Metode":<25} {"Accuracy":>9} {"F1":>9} {"AUC":>9} {"Time":>8}')
print('-' * 60)
for idx, row in results_df_adv.iterrows():
    t = row.get('Time', 0)
    print(f'{idx:<25} {row["Accuracy"]:>9.4f} {row["F1-Score"]:>9.4f} {row["ROC-AUC"]:>9.4f} {t:>7.1f}s')
print('-' * 60)
best_auc = results_df_adv['ROC-AUC'].max()
best_method = results_df_adv['ROC-AUC'].idxmax()
print(f'\n  MODEL TERBAIK: {best_method} (AUC = {best_auc:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
x = np.arange(len(results_adv))
width = 0.2
for i, metric in enumerate(['Accuracy', 'Precision', 'Recall', 'F1-Score']):
    vals = [results_adv[m][metric] for m in results_adv]
    ax.bar(x + i * width, vals, width, label=metric)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(list(results_adv.keys()), fontsize=8, rotation=15)
ax.set_ylabel('Score')
ax.set_title('Perbandingan Metrics', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
roc_vals = [results_adv[m]['ROC-AUC'] for m in results_adv]
colors_alg = plt.cm.Set2(np.linspace(0, 1, len(results_adv)))
bars = ax.bar(range(len(results_adv)), roc_vals, color=colors_alg, width=0.6, edgecolor='black')
for bar, val in zip(bars, roc_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(results_adv)))
ax.set_xticklabels(list(results_adv.keys()), fontsize=8, rotation=15)
ax.set_ylabel('ROC-AUC')
ax.set_title('ROC-AUC Comparison', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Random (0.5)')
ax.legend()

plt.suptitle('PERBANDINGAN SOLUSI LANJUTAN - Survival_Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('advanced_solutions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

results_df_adv.to_csv('advanced_solutions_results.csv')
print('  => Saved: advanced_solutions_results.csv')

In [ ]:
# ================================================================
# KESIMPULAN AKHIR
# ================================================================
print('=' * 80)
print('KESIMPULAN AKHIR - TUGAS AKHIR')
print('=' * 80)

print('TAHAPAN YANG TELAH DILAKUKAN:')
print('')
print('  BAGIAN 1: DATA PROCESSING')
print('   - Cleaning dataset (duplikat, whitespace, validasi)')
print('   - Handling missing values (median/modus imputation)')
print('   - Outlier removal (IQR method)')
print('   - Normalisasi (Min-Max) & Standarisasi (Z-Score)')
print('   - One-Hot Encoding untuk fitur kategorikal')

print('  BAGIAN 2: EXPLORATORY DATA ANALYSIS (EDA)')
print('   - Data quality check')
print('   - Univariate analysis (numerik & kategorikal)')
print('   - Bivariate analysis (vs Survival_5_years)')
print('   - Chi-Square test')
print('   - Correlation matrix')
print('   - Risk factors & demographic analysis')

print('  BAGIAN 3: TARGET ANALYSIS')
print('   - Perbandingan 7 target candidates')
print('   - RF evaluation untuk setiap target')
print('   - Identifikasi target paling prediktif')

print('  BAGIAN 4: TRAINING IMPORTANCE FITUR')
print('   - Feature Importance (RF + Mutual Information)')
print('   - Seleksi fitur intersection RF & MI')
print('   - SMOTE K=5 untuk balancing')
print('   - Training 4 model: SVM, XGBoost, RF, KNN + Baseline')
print('   - Confusion Matrix, ROC Curves, Prediction Table')

print('  BAGIAN 5: ADVANCED SOLUTIONS')
print('   - Neural Network (MLP)')
print('   - Polynomial Features + RF')
print('   - PCA + XGBoost')
print('   - Stacking Ensemble')
print('   - GridSearch XGBoost')

best_model = results_df_fi['ROC-AUC'].idxmax()
best_auc_val = results_df_fi['ROC-AUC'].max()
best_adv = results_df_adv['ROC-AUC'].idxmax()
best_adv_auc = results_df_adv['ROC-AUC'].max()

print(f'\n  MODEL TERBAIK (Bagian 4): {best_model} (AUC = {best_auc_val:.4f})')
print(f'  MODEL TERBAIK (Bagian 5): {best_adv} (AUC = {best_adv_auc:.4f})')

print(f'\n{"="*80}')
print('TUGAS AKHIR SELESAI!')
print('Semua output tersimpan di folder: preprocessed/, eda_output/, Training_Importance_Fitur/')
print(f'{"="*80}')